# Image Memory Chatbot (Notebook)

This notebook lets you upload/select images and ask questions about them, such as:
- What white dresses did I save?
- What places have I wanted to go this semester?
- What themes show up in my friend's and my journal entries?

It uses a vision-capable OpenAI model and answers using only image evidence.

In [1]:
# If needed, uncomment and run:
# %pip install openai pillow

import base64
import os
from pathlib import Path
from typing import List

import openai
from IPython.display import display
from PIL import Image

In [2]:
from openai import OpenAI

In [3]:
print(openai.__version__)

2.32.0


In [4]:
# Secure key entry (runtime only; not saved in plain text)
import os
from getpass import getpass

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter OPENAI_API_KEY: ")
    print("OPENAI_API_KEY set for this notebook session.")
else:
    print("OPENAI_API_KEY already set in environment.")

Enter OPENAI_API_KEY:  ········


OPENAI_API_KEY set for this notebook session.


In [5]:
MODEL_NAME = "gpt-4.1-mini"


def _encode_image_file(image_path: Path) -> str:
    return base64.b64encode(image_path.read_bytes()).decode("utf-8")


def _build_prompt(question: str) -> str:
    return (
        "You are an image-aware personal archive assistant.\n"
        "The user asks memory queries based on uploaded/supplied images.\n"
        "Answer only from visual evidence in the images.\n"
        "If evidence is weak or missing, state uncertainty clearly.\n"
        "When possible, mention image numbers that support claims.\n\n"
        f"User question: {question}"
    )


def answer_with_vision(question: str, image_paths: List[Path]) -> str:
    api_key = os.getenv("OPENAI_API_KEY")
    if not api_key:
        return "Set OPENAI_API_KEY before running this cell."

    client = OpenAI(api_key=api_key)

    content = [{"type": "input_text", "text": _build_prompt(question)}]
    for path in image_paths:
        b64 = _encode_image_file(path)
        content.append(
            {
                "type": "input_image",
                "image_url": f"data:image/jpeg;base64,{b64}",
            }
        )

    response = client.responses.create(
        model=MODEL_NAME,
        input=[{"role": "user", "content": content}],
    )
    return response.output_text

## Upload images directly in notebook

Use the uploader below, then run the next cell to ask a question about the uploaded images.

In [6]:
import tempfile
from pathlib import Path

import ipywidgets as widgets
from IPython.display import clear_output, display
from PIL import Image

uploader = widgets.FileUpload(
    accept=".png,.jpg,.jpeg,.webp",
    multiple=True,
    description="Upload images",
)
helper_text = widgets.HTML(
    value="<b>Tip:</b> You can upload multiple files at once. On Mac file picker, use Cmd-click or Shift-click."
)
select_button = widgets.Button(description="Pick files from computer")
question_box = widgets.Text(
    value="",
    description="Question:",
    layout=widgets.Layout(width="90%"),
)
run_button = widgets.Button(description="Analyze", button_style="primary")
output = widgets.Output()

picked_paths: list[Path] = []


def _save_uploads_to_temp(upload_value) -> list[Path]:
    temp_paths = []

    if hasattr(upload_value, "values"):
        items = list(upload_value.values())
    else:
        items = list(upload_value)

    for idx, item in enumerate(items, start=1):
        if isinstance(item, dict):
            name = item.get("name") or item.get("metadata", {}).get("name") or f"upload_{idx}.jpg"
            content = item.get("content") or item.get("data")
        else:
            name = f"upload_{idx}.jpg"
            content = None

        if content is None:
            raise ValueError("Could not read uploaded file bytes. Please re-upload and try again.")

        suffix = Path(name).suffix or ".jpg"
        tmp = tempfile.NamedTemporaryFile(delete=False, suffix=suffix)
        tmp.write(content)
        tmp.flush()
        tmp.close()
        temp_paths.append(Path(tmp.name))

    return temp_paths


def on_pick_clicked(_):
    with output:
        try:
            import tkinter as tk
            from tkinter import filedialog

            root = tk.Tk()
            root.withdraw()
            files = filedialog.askopenfilenames(
                title="Select one or more images",
                filetypes=[("Images", "*.png *.jpg *.jpeg *.webp")],
            )
            root.destroy()

            if files:
                picked_paths.clear()
                picked_paths.extend(Path(f) for f in files)
                print(f"Picked {len(picked_paths)} file(s) from computer.")
            else:
                print("No files selected.")
        except Exception as exc:
            print(f"Native picker unavailable in this environment: {exc}")


def on_run_clicked(_):
    with output:
        clear_output()
        if not question_box.value.strip():
            print("Please enter a question.")
            return

        image_paths = []

        if uploader.value:
            image_paths.extend(_save_uploads_to_temp(uploader.value))

        if picked_paths:
            image_paths.extend(picked_paths)

        if not image_paths:
            print("Please upload or pick at least one image.")
            return

        print(f"Using {len(image_paths)} image(s).")

        for i, p in enumerate(image_paths, start=1):
            try:
                print(f"Image {i}: {p.name}")
                display(Image.open(p))
            except Exception:
                print(f"Image {i}: (preview unavailable)")

        answer = answer_with_vision(question_box.value.strip(), image_paths)
        print("\nAnswer:\n")
        print(answer)


select_button.on_click(on_pick_clicked)
run_button.on_click(on_run_clicked)

display(widgets.VBox([helper_text, uploader, select_button, question_box, run_button, output]))

## Notes

- Upload one or more images using the widget; no default files are used.
- For large batches, keep image counts moderate for faster responses.
- If you want true drag-and-drop uploads inside notebook, I can add an `ipywidgets.FileUpload` version next.